# 03 - Radiomics feature estraction

In [ ]:
# Thread limits must be set BEFORE importing numpy / SimpleITK / radiomics,
# otherwise pyradiomics' parallel BLAS calls can race and crash long batch runs.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS", "1")

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import gc
import logging
import traceback

import cv2
import numpy as np
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm

import radiomics
from radiomics import featureextractor

sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(1)
radiomics.setVerbosity(logging.ERROR)

from src.paths import ISIC_IMAGES_DIR, ISIC_MASKS_DIR, RESULTS_DIR, ensure_dir

FEATURES_DIR = ensure_dir(RESULTS_DIR / "features")
print(f"Images:    {ISIC_IMAGES_DIR}")
print(f"Masks:     {ISIC_MASKS_DIR}")
print(f"Features → {FEATURES_DIR}")

## Input validation

Intersect the images and masks directories to identify pairs we can actually process. We skip any image without a matching mask (and vice versa) — those would silently break the extractor or produce empty rows.

In [ ]:
image_files = {os.path.splitext(f)[0]: f for f in os.listdir(ISIC_IMAGES_DIR)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))}
mask_files  = {os.path.splitext(f)[0]: f for f in os.listdir(ISIC_MASKS_DIR)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))}

pair_ids = sorted(set(image_files) & set(mask_files))

print(f"Images:        {len(image_files):,}")
print(f"Masks:         {len(mask_files):,}")
print(f"Matched pairs: {len(pair_ids):,}")
print(f"Image-only:    {len(set(image_files) - set(mask_files)):,}")
print(f"Mask-only:     {len(set(mask_files) - set(image_files)):,}")

## Feature extractor configuration

In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.disableAllFeatures()

features = ["firstorder", "glcm", "gldm", "glrlm", "glszm", "ngtdm", "shape2D"]
for feature in features:
    extractor.enableFeatureClassByName(feature)

print(f"Enabled feature classes: {features}")

## Helper functions

In [ ]:
def to_scalar(value):
    if isinstance(value, (list, tuple, set)):
        return next(iter(value))
    if isinstance(value, dict):
        return next(iter(value.values()))
    return value

In [ ]:
def extract_all_channels(image_path, mask_path, extractor):
    """Extract pyradiomics features for the R, G, B and gray channels of one image.

    Reads the image and mask ONCE from disk and reuses the in-memory SimpleITK
    objects across all four channels — no .nii.gz intermediates, ~4x less disk
    and ~4x less I/O than the original per-channel pipeline.

    Returns a flat dict with channel-prefixed keys:
        {'blue__firstorder_Mean': ..., 'red__glcm_Joint...': ..., ...}
    """
    img_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    mask_np = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask_np is None:
        raise ValueError(f"Could not read mask: {mask_path}")
    if mask_np.max() == 0:
        raise ValueError("Empty mask (all zeros)")

    mask_bin  = (mask_np > 127).astype(np.uint8)
    mask_sitk = sitk.GetImageFromArray(mask_bin)

    channels = {
        "blue":  img_bgr[:, :, 0],
        "green": img_bgr[:, :, 1],
        "red":   img_bgr[:, :, 2],
        "gray":  cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY),
    }

    out = {}
    for ch_name, arr in channels.items():
        ch_sitk = sitk.GetImageFromArray(arr)
        raw = extractor.execute(ch_sitk, mask_sitk)
        for k, v in raw.items():
            if "diagnostics_" in k:
                continue
            out[f"{ch_name}__{k}"] = to_scalar(v)

    return out


def get_processed_ids(csv_path):
    """Read only the image_id column of the output CSV — used to resume runs."""
    if not csv_path.exists():
        return set()
    try:
        return set(pd.read_csv(csv_path, usecols=["image_id"])["image_id"].astype(str))
    except (ValueError, KeyError, pd.errors.EmptyDataError):
        return set()

## Extract features

Single pass over the matched pairs. For each image we read once, split into R/G/B/gray, run pyradiomics 4 times against the same in-memory mask, and append the flattened row to a buffer. Every `BATCH_SIZE` rows the buffer is flushed to CSV and freed — RAM stays roughly constant regardless of dataset size.

The output CSV doubles as the resume checkpoint: re-running this cell skips any `image_id` already present in `features_all_channels.csv`.

In [ ]:
out_csv  = FEATURES_DIR / "features_all_channels.csv"
log_path = FEATURES_DIR / "errors.log"
BATCH_SIZE = 100  # flush every N rows

processed  = get_processed_ids(out_csv)
to_process = [pid for pid in pair_ids if pid not in processed]
print(f"Already processed: {len(processed):,}")
print(f"To process:        {len(to_process):,}")

buffer = []
n_errors = 0

for image_id in tqdm(to_process, desc="Extracting"):
    try:
        img_path = ISIC_IMAGES_DIR / image_files[image_id]
        msk_path = ISIC_MASKS_DIR / mask_files[image_id]
        features = extract_all_channels(img_path, msk_path, extractor)
        features["image_id"] = image_id
        buffer.append(features)
    except Exception:
        n_errors += 1
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"\n--- image_id={image_id} ---\n")
            f.write(traceback.format_exc() + "\n")

    if len(buffer) >= BATCH_SIZE:
        df = pd.DataFrame(buffer).set_index("image_id")
        df.to_csv(out_csv, mode="a", header=not out_csv.exists(), index=True)
        buffer.clear()
        gc.collect()

# Final flush
if buffer:
    df = pd.DataFrame(buffer).set_index("image_id")
    df.to_csv(out_csv, mode="a", header=not out_csv.exists(), index=True)
    buffer.clear()

print(f"\nDone. Errors: {n_errors} (see {log_path})")

## Save as Parquet

Append-mode CSV is the right format for the streaming write (any text editor can recover from a half-written line), but it bloats on disk and is slow to read. Convert the final result to Parquet once — typically 5–10× smaller, ~10× faster to load, and preserves dtypes.

In [ ]:
parquet_path = FEATURES_DIR / "features_all_channels.parquet"

# Read the streamed CSV back in one shot (this is the only point where the
# whole table sits in RAM — for 75k × ~500 cols of float64 that's ~300 MB,
# fine on the VM but reduce BATCH_SIZE if it's a problem locally).
df = pd.read_csv(out_csv, index_col="image_id")
df.to_parquet(parquet_path, compression="snappy")

print(f"Saved {parquet_path} ({parquet_path.stat().st_size / 1024**2:.1f} MB)")
print(f"Shape:                 {df.shape}")
print(f"Columns with any NaN:  {(df.isna().any()).sum()}")
print(f"Rows with any NaN:     {(df.isna().any(axis=1)).sum()}")
df.head()